ALML CA1 

PART A: Predicting Factory Machine Status (50 marks)

Importing Pandas and the Machine Learning Models (Random Forest, and KNearestNeighbors), and loading the dataset factory_data.csv onto df

In [21]:
import pandas as pd

from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier

df = pd.read_csv("datasets/factory_data.csv")

In the dataset factory_data.csv there missing entries in the following columns: 

- Quality: 991 

- Process T (C): 400

- Rotation Speed (rpm): 1,188



I imputed this missing data by: 

- Finding the modal value for Quality

- Finding the mean value for Process T (C) and Rotation Speed (rpm) respectively

In [22]:
df.fillna({
    'Quality': df['Quality'].mode()[0],
    'Process T (C)': df['Process T (C)'].mean(),
    'Rotation Speed (rpm)': df['Rotation Speed (rpm)'].mean()
}, inplace=True)

Since we have catagorical values in our Quality column (M, L, H), we need to use Label Encoding to convert it into a numeric form in df

In [23]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
for col in df.select_dtypes(include=["object"]).columns:
    df[col] = le.fit_transform(df[col])

I am now going to define the Features (X) and the Target (y) 

- X will include the Quality, Ambient T, Process T, Rotation Speed, Torque and Tool Wear columns

- y will include only the Machine Status column

In [24]:
X = df.drop(columns=["Unique ID", "Product ID", "Machine Status"])
y = df["Machine Status"]

I will now split the dataset such that 80% of it will be used to train the model, while 20% of it will be used for testing the model

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

Training a Random Forest model, setting n_estimators as 100

In [26]:
rf_model = RandomForestClassifier(
    n_estimators=100, 
    random_state=42
)
rf_model.fit(X_train, y_train)
 
# Make predictions
rf_pred = rf_model.predict(X_test)

Training a K Nearest Neighbors model, setting n_neighbors as 6

In [27]:
knn_model = KNeighborsClassifier(n_neighbors = 6)
knn_model.fit(X_train, y_train)
 
# Make predictions
knn_pred = knn_model.predict(X_test)

After training, I will compare the accuracy and classification report of both models to determine the better model

In [28]:
from sklearn.metrics import classification_report, accuracy_score

# Random Forest
print("RF Accuracy:", accuracy_score(y_test, rf_pred))
print("\nClassification Report:\n", classification_report(y_test, rf_pred))

print("------------------------------------------------------")

# K Nearest Neighbors
print("KNN Accuracy:", accuracy_score(y_test, knn_pred))
print("\nClassification Report:\n", classification_report(y_test, knn_pred))

RF Accuracy: 0.9915

Classification Report:
               precision    recall  f1-score   support

           0       0.99      1.00      1.00      3864
           1       1.00      0.75      0.86       136

    accuracy                           0.99      4000
   macro avg       1.00      0.88      0.93      4000
weighted avg       0.99      0.99      0.99      4000

------------------------------------------------------
KNN Accuracy: 0.9715

Classification Report:
               precision    recall  f1-score   support

           0       0.97      1.00      0.99      3864
           1       0.81      0.21      0.34       136

    accuracy                           0.97      4000
   macro avg       0.89      0.61      0.66      4000
weighted avg       0.97      0.97      0.96      4000



We can see that both models Random Forest and K Nearest Neighbors have a high accuracy of 99% and 97% respectively. However this is misleading as majority of the test data consists of 0, which can lead to a high accuracy should the model consider everything as 0. Which is why we will need to use recall to compare each models performance.

The misleading accuracy is what happened to K Nearest Neighbors due to its poor recall of 0.21 for 1, and having a recall of 1.00 for 0. Random Forest on the other hand had a much higher recall of 0.75 for 1, and also had a recall of 1.00 for 0. 

Therefore, by comparing recall value for 1, Random Forest is the better model in determining Machine Status.